In [1]:
from pathlib import Path
import hashlib
import pandas as pd

INPUT_DIR = Path("/kaggle/input")
RESULTS_DIR = Path("/kaggle/working/artifacts/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)


def find_one(filename):
    matches = list(INPUT_DIR.rglob(filename))

    if len(matches) != 1:
        raise RuntimeError(
            f"Expected one {filename}, found {len(matches)}: {matches}"
        )

    return matches[0]


# Verify the original dataset matches the previously audited labels.
dataset_dir = find_one("train.txt").parent

expected_hashes = {
    "train.txt": (
        "52af9a4fdc769396bc5ce530872cf0d51bce"
        "8fd7e25579b333007d4ebee6a6ba"
    ),
    "test.txt": (
        "7c92ae0bef49652005e1541a1f2b774e31"
        "3be4431a891cb0aa5539d37ca2a71e"
    ),
}

for filename, expected in expected_hashes.items():
    content = (dataset_dir / filename).read_bytes()
    actual = hashlib.sha256(content).hexdigest()

    assert actual == expected, f"Label checksum mismatch: {filename}"

# Load saved splits while preserving patient identifiers as strings.
training = pd.read_csv(
    find_one("train.csv"),
    dtype=str,
    keep_default_na=False,
)
validation = pd.read_csv(
    find_one("validation.csv"),
    dtype=str,
    keep_default_na=False,
)

required_columns = {
    "filename", "label", "patient_id",
    "source", "sha256", "split_group",
}

for name, table, expected_count in [
    ("Training", training, 23873),
    ("Validation", validation, 6103),
]:
    assert required_columns.issubset(table.columns)
    assert len(table) == expected_count
    assert table["filename"].is_unique
    assert table["sha256"].is_unique
    assert set(table["label"]) == {"negative", "positive"}

    missing = [
        filename
        for filename in table["filename"]
        if not (dataset_dir / "train" / filename).is_file()
    ]

    assert not missing, f"{name}: missing images: {missing[:5]}"
    print(f"{name} images: {len(table)}")

for column in ["filename", "sha256", "split_group"]:
    assert set(training[column]).isdisjoint(validation[column]), (
        f"Train/validation overlap: {column}"
    )

print("Dataset label checksums match.")
print("Saved split and image-path checks passed.")

Training images: 23873
Validation images: 6103
Dataset label checksums match.
Saved split and image-path checks passed.


In [2]:
import sys
import inspect
from pathlib import Path

# Locate the shared preprocessing module in the attached source.
module_files = list(
    Path("/kaggle/input").glob("**/src/covid_xray/data.py")
)

if len(module_files) != 1:
    raise RuntimeError(
        f"Expected one preprocessing module, found: {module_files}"
    )

source_dir = module_files[0].parents[1]
sys.path.insert(0, str(source_dir))

from covid_xray.data import preprocess_image

loaded_path = Path(inspect.getfile(preprocess_image)).resolve()
assert loaded_path == module_files[0].resolve()

print("Shared preprocessing function loaded.")
print("Source:", loaded_path)

Shared preprocessing function loaded.
Source: /kaggle/input/datasets/umangkumar8755875/chest-xray-project-source/Chest-X-Ray-Diagnosis-using-Deep-Learning--main/src/covid_xray/data.py


In [3]:
import numpy as np
import tensorflow as tf

print("TensorFlow:", tf.__version__)

# Verify CPU execution before building the input pipeline.
with tf.device("/CPU:0"):
    result = tf.reduce_sum(tf.constant([1.0, 2.0, 3.0]))

assert float(result.numpy()) == 6.0
print("TensorFlow CPU check passed.")

TensorFlow: 2.20.0
TensorFlow CPU check passed.


2026-10-18 11:10:40.463032: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [4]:
BATCH_SIZE = 32
SEED = 42

# Keep the class mapping explicit and consistent.
LABEL_MAP = {"negative": 0, "positive": 1}


def load_image(path):
    # Reuse the same Pillow preprocessing as the application.
    filename = path.numpy().decode("utf-8")
    return preprocess_image(filename)


def prepare_example(path, label):
    image = tf.py_function(
        func=load_image,
        inp=[path],
        Tout=tf.float32,
    )
    image.set_shape((224, 224, 1))

    label = tf.reshape(tf.cast(label, tf.float32), (1,))
    return image, label


def make_dataset(table, training_mode=False):
    paths = [
        str(dataset_dir / "train" / filename)
        for filename in table["filename"]
    ]
    labels = table["label"].map(LABEL_MAP)

    if labels.isna().any():
        raise ValueError("Unknown class label.")

    dataset = tf.data.Dataset.from_tensor_slices(
        (paths, labels.to_numpy(dtype=np.float32))
    )

    # Shuffle paths before decoding to keep memory usage modest.
    if training_mode:
        dataset = dataset.shuffle(
            buffer_size=len(table),
            seed=SEED,
            reshuffle_each_iteration=True,
        )

    dataset = dataset.map(
        prepare_example,
        num_parallel_calls=2,
        deterministic=True,
    )

    return dataset.batch(BATCH_SIZE).prefetch(1)


train_ds = make_dataset(training, training_mode=True)
val_ds = make_dataset(validation)

# Decode one actual batch and verify its contents.
images, labels = next(iter(train_ds))

tf.debugging.assert_shapes([
    (images, (32, 224, 224, 1)),
    (labels, (32, 1)),
])
tf.debugging.assert_all_finite(images, "Non-finite image pixels")
tf.debugging.assert_greater_equal(images, 0.0)
tf.debugging.assert_less_equal(images, 1.0)
tf.debugging.assert_equal(
    tf.reduce_all((labels == 0.0) | (labels == 1.0)),
    True,
)

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Training batches:", int(tf.data.experimental.cardinality(train_ds)))
print("Validation batches:", int(tf.data.experimental.cardinality(val_ds)))
print("Actual training batch check passed.")

Image batch shape: (32, 224, 224, 1)
Label batch shape: (32, 1)
Training batches: 747
Validation batches: 191
Actual training batch check passed.


In [5]:
val_images, val_labels = next(iter(val_ds))

tf.debugging.assert_shapes([
    (val_images, (32, 224, 224, 1)),
    (val_labels, (32, 1)),
])
tf.debugging.assert_all_finite(
    val_images, "Non-finite validation pixels"
)
tf.debugging.assert_greater_equal(val_images, 0.0)
tf.debugging.assert_less_equal(val_images, 1.0)
tf.debugging.assert_equal(
    tf.reduce_all((val_labels == 0.0) | (val_labels == 1.0)),
    True,
)

# Verify validation labels preserve manifest order.
expected_labels = (
    validation["label"]
    .map(LABEL_MAP)
    .iloc[:32]
    .to_numpy(dtype=np.float32)
    .reshape(-1, 1)
)

np.testing.assert_array_equal(
    val_labels.numpy(),
    expected_labels,
)

print("Validation image shape:", val_images.shape)
print("Validation label shape:", val_labels.shape)
print("Validation batch and label-order checks passed.")

Validation image shape: (32, 224, 224, 1)
Validation label shape: (32, 1)
Validation batch and label-order checks passed.


## Preprocessing and Batch Pipeline

The notebook imports the shared preprocessing function from
the attached project source.

Images are converted to grayscale, resized with aspect-ratio
preservation and black padding, and normalized to [0, 1].
Output shape is (224, 224, 1), with float32 pixels.

Class mapping:
- negative: 0
- positive: 1

The pipeline uses batches of 32:
- Training: 23,873 images, 747 batches.
- Validation: 6,103 images, 191 batches.

Training paths are shuffled before decoding. Validation order
is preserved. The final batch is retained even when incomplete.

One actual batch from each split passed shape, finite-value,
pixel-range, and binary-label checks. Validation labels also
matched manifest order.

Full pipeline iteration and model training have not yet been
performed. Augmentation has not yet been added.